# CFS Wall Stud Design
Single-floor wall stud design using RSG CFS15 API (win32com).  
Matches Simpson CFS workflow for stacked wall design validation.

## 1. Setup & COM Connection

In [ ]:
import win32com.client
import pandas as pd
from IPython.display import display

# ── CFS version: change to 'RSG.CFS14.Calculation' if using CFS14 ──
CFS_PROGID = 'RSG.CFS15.Calculation'
SPEC       = 42   # AISI2024USASD  (37 = AISI2022USASD, 27 = AISI2016USASD)

# Singleton – reuse across all section checks
_cfs = None

def get_cfs():
    global _cfs
    if _cfs is None:
        _cfs = win32com.client.Dispatch(CFS_PROGID)
        if not _cfs.HasLicense:
            raise RuntimeError('CFS license not available')
        print(f'Connected: {CFS_PROGID}')
    return _cfs

cfs = get_cfs()

## 2. Load-path helpers

**Axial (gravity):**  
`PLF = PSF × trib_ft` → `P_stud (k) = PLF × spacing_ft / 1000`

**Lateral (wind/seismic normal to wall):**  
`w_stud (lb/ft) = PSF × spacing_ft` → produces a uniform line load on the stud  
→ `M_x (k·in) = w × H² / 8` (simply supported)  
→ `V_y (k)    = w × H / 2`

In [ ]:
def axial_plf(psf, trib_ft):
    """PSF area load on floor/roof → PLF delivered to the wall top plate."""
    return psf * trib_ft                   # lb/ft along wall

def axial_stud(plf, spacing_in):
    """PLF along wall → axial force per stud (kips)."""
    spacing_ft = spacing_in / 12
    return plf * spacing_ft / 1000         # kips

def lateral_stud_moment(psf, spacing_in, height_ft):
    """Lateral PSF (wind/seismic) → Mx (k·in) at mid-span of stud (simply supported)."""
    spacing_ft = spacing_in / 12
    w = psf * spacing_ft / 1000            # k/ft  (uniform load on stud)
    Mx_kft = w * height_ft**2 / 8         # k·ft  (simply supported)
    return Mx_kft * 12                     # k·in

def lateral_stud_shear(psf, spacing_in, height_ft):
    """Lateral PSF → Vy (kips) at end of stud."""
    spacing_ft = spacing_in / 12
    w = psf * spacing_ft / 1000            # k/ft
    return w * height_ft / 2              # kips

# ── Quick sanity check ──
print('PLF check : 10 psf × 10 ft =', axial_plf(10, 10), 'lb/ft  (expect 100)')
print('Axial check: 100 plf, 16in =', round(axial_stud(100, 16),4), 'k  (expect 0.1333)')
print('Moment check: 10 psf, 16in, 10ft =', round(lateral_stud_moment(10, 16, 10),2), 'k·in')
print('Shear  check: 10 psf, 16in, 10ft =', round(lateral_stud_shear(10, 16, 10),4), 'k')

## 3. Unbraced length calculator

| Config | Ly = Lt = Lm |
|---|---|
| none | H (full height) |
| midpoint | H/2 |
| third | H/3 |
| 48in / 60in / 72in | fixed spacing |
| custom | H/n |

Lx is always full height (strong axis – wall sheathing does not brace strong axis buckling).

In [ ]:
def unbraced_lengths(height_ft, brace_config='third', custom_n=None):
    """
    Returns dict with Lx, Ly, Lt, Lm (all in inches).
    brace_config: 'none' | 'midpoint' | 'third' | '48in' | '60in' | '72in' | 'custom'
    custom_n: number of equal brace spacings (used when brace_config='custom')
    """
    H_in = height_ft * 12

    brace_map = {
        'none':     H_in,
        'midpoint': H_in / 2,
        'third':    H_in / 3,
        '48in':     48.0,
        '60in':     60.0,
        '72in':     72.0,
    }

    if brace_config == 'custom':
        if custom_n is None or custom_n < 1:
            raise ValueError('custom_n must be >= 1 when brace_config="custom"')
        Ly = H_in / custom_n
    elif brace_config in brace_map:
        Ly = brace_map[brace_config]
    else:
        raise ValueError(f'Unknown brace_config: {brace_config}. '
                         f'Use: none, midpoint, third, 48in, 60in, 72in, custom')

    return {'Lx': H_in, 'Ly': Ly, 'Lt': Ly, 'Lm': Ly}


print('third-pt, 10ft:', unbraced_lengths(10, 'third'))
print('48in,     10ft:', unbraced_lengths(10, '48in'))
print('custom/4, 10ft:', unbraced_lengths(10, 'custom', custom_n=4))

## 4. ASD Load combinations

Per ASCE 7 (table from Simpson CFS reference).  
Each combo returns `(P_k, Mx_kin, Vy_k)` for the stud.

| # | Formula |
|---|---|
| 1 | D + L |
| 2 | D + (Lr or S) |
| 3 | D + 0.75L + 0.75(Lr or S) + 0.75·W_mwfw |
| 4 | D + 0.75L + 0.75(Lr or S) + 0.75·W_mwfl |
| 5 | D + W_cc_w |
| 6 | D + W_cc_l |
| 7 | D + E (Eh) |
| 8 | D + 0.75L + 0.75S + 0.75E |
| 9 | 0.6D + W_uplift |
| 10 | 0.6D + W_cc_l |

In [ ]:
def build_load_combinations(loads, geometry):
    """
    Build ASD load combos. Returns list of dicts:
      [{'combo': int, 'label': str, 'P_k': float, 'Mx_kin': float, 'Vy_k': float}, ...]

    loads dict keys (all in PSF unless noted):
      D_wall      – wall self-weight
      D_A, L_A    – dead/live from Side A floor or roof
      D_B, L_B    – dead/live from Side B (interior walls only)
      S           – roof snow
      W_uplift    – roof wind uplift (acts opposite gravity, negative axial)
      W_mwfw      – MWFRS windward lateral
      W_mwfl      – MWFRS leeward lateral
      W_cc_w      – C&C windward lateral
      W_cc_l      – C&C leeward lateral
      L_lat       – interior lateral live (e.g. partition impact)
      Eh          – seismic horizontal coefficient (fraction of D_axial)
      Ev          – seismic vertical coefficient

    geometry dict keys:
      height_ft   – wall height
      spacing_in  – stud spacing
      trib_A_ft   – tributary length side A
      trib_B_ft   – tributary length side B (0 if exterior)
      wall_type   – 'exterior' | 'interior'
    """
    H   = geometry['height_ft']
    sp  = geometry['spacing_in']
    tA  = geometry['trib_A_ft']
    tB  = geometry.get('trib_B_ft', 0.0)
    wt  = geometry['wall_type']      # 'exterior' | 'interior'

    # ── Gravity PLF along wall → axial stud force (kips) ──
    def P(D_psf_A, D_psf_B=0, L_psf_A=0, L_psf_B=0, S_psf=0,
          uplift_psf=0, Ev_frac=0, D_coeff=1.0, L_coeff=1.0,
          S_coeff=1.0, uplift_coeff=0.0):
        """Total axial stud force (kips, compression positive)."""
        plf_D  = axial_plf(loads['D_wall'], H/2) * 2  # self-weight: tributary is stud spacing in plan
        # Actually wall self-weight: PLF = D_wall_psf * stud_spacing_ft, axial = D_wall * H * spacing_ft
        # Simpler: wall dead = psf * wall_height * spacing_ft
        P_wall = loads['D_wall'] * H * (sp/12) / 1000

        P_DA   = axial_stud(axial_plf(D_psf_A, tA), sp)
        P_DB   = axial_stud(axial_plf(D_psf_B, tB), sp) if wt == 'interior' else 0
        P_LA   = axial_stud(axial_plf(L_psf_A, tA), sp)
        P_LB   = axial_stud(axial_plf(L_psf_B, tB), sp) if wt == 'interior' else 0
        P_S    = axial_stud(axial_plf(S_psf,   tA), sp)
        P_up   = axial_stud(axial_plf(uplift_psf, tA), sp)  # negative (uplift)

        P_total = (D_coeff * (P_wall + P_DA + P_DB)
                   + L_coeff * (P_LA + P_LB)
                   + S_coeff * P_S
                   - uplift_coeff * P_up)
        return P_total

    # ── Lateral load → Mx and Vy per stud ──
    def MV(lat_psf, factor=1.0):
        Mx = lateral_stud_moment(lat_psf * factor, sp, H)
        Vy = lateral_stud_shear( lat_psf * factor, sp, H)
        return Mx, Vy

    D  = loads
    combos = []

    def add(n, label, P_k, Mx_kin, Vy_k):
        combos.append({'combo': n, 'label': label,
                       'P_k': round(P_k, 4),
                       'Mx_kin': round(Mx_kin, 4),
                       'Vy_k': round(Vy_k, 4)})

    # 1: D + L
    Mx1, Vy1 = MV(D.get('L_lat', 0)) if wt == 'interior' else (0, 0)
    add(1, 'D + L',
        P(D['D_A'], D.get('D_B',0), D['L_A'], D.get('L_B',0)),
        Mx1, Vy1)

    # 2: D + (Lr or S)
    add(2, 'D + (Lr or S)',
        P(D['D_A'], D.get('D_B',0), S_psf=D['S']),
        0, 0)

    # 3: D + 0.75L + 0.75S + 0.75·W_mwfw
    Mx3, Vy3 = MV(D.get('W_mwfw', 0), 0.75)
    add(3, 'D + 0.75L + 0.75S + 0.75·W_mwfw',
        P(D['D_A'], D.get('D_B',0), D['L_A'], D.get('L_B',0),
          S_psf=D['S'], L_coeff=0.75, S_coeff=0.75),
        Mx3, Vy3)

    # 4: D + 0.75L + 0.75S + 0.75·W_mwfl
    Mx4, Vy4 = MV(D.get('W_mwfl', 0), 0.75)
    add(4, 'D + 0.75L + 0.75S + 0.75·W_mwfl',
        P(D['D_A'], D.get('D_B',0), D['L_A'], D.get('L_B',0),
          S_psf=D['S'], L_coeff=0.75, S_coeff=0.75),
        Mx4, Vy4)

    # 5: D + W_cc_w
    Mx5, Vy5 = MV(D.get('W_cc_w', 0))
    add(5, 'D + W_cc_w',
        P(D['D_A'], D.get('D_B',0)),
        Mx5, Vy5)

    # 6: D + W_cc_l
    Mx6, Vy6 = MV(D.get('W_cc_l', 0))
    add(6, 'D + W_cc_l',
        P(D['D_A'], D.get('D_B',0)),
        Mx6, Vy6)

    # 7: D + E  (seismic: Eh as fraction of total dead axial)
    P7  = P(D['D_A'], D.get('D_B',0), Ev_frac=D.get('Ev',0))
    # Seismic lateral: treat Eh as equivalent PSF = Eh × D_wall (simplified)
    lat_E = D.get('Eh', 0) * D['D_wall']
    Mx7, Vy7 = MV(lat_E)
    add(7, 'D + E', P7, Mx7, Vy7)

    # 8: D + 0.75L + 0.75S + 0.75E
    Mx8, Vy8 = MV(lat_E, 0.75)
    add(8, 'D + 0.75L + 0.75S + 0.75E',
        P(D['D_A'], D.get('D_B',0), D['L_A'], D.get('L_B',0),
          S_psf=D['S'], L_coeff=0.75, S_coeff=0.75),
        Mx8, Vy8)

    # 9: 0.6D + W_uplift  (tension / uplift check)
    Mx9, Vy9 = MV(D.get('W_mwfw', 0))
    add(9, '0.6D + W_uplift',
        P(D['D_A'], D.get('D_B',0), uplift_psf=D.get('W_uplift',0),
          D_coeff=0.6, uplift_coeff=1.0),
        Mx9, Vy9)

    # 10: 0.6D + W_cc_l
    Mx10, Vy10 = MV(D.get('W_cc_l', 0))
    add(10, '0.6D + W_cc_l',
        P(D['D_A'], D.get('D_B',0), D_coeff=0.6),
        Mx10, Vy10)

    return combos


# ── Preview combo table ──
print('Combo table preview (not yet connected to actual loads)')

## 5. Core: single-stud check

Loads a section, runs `MemberCheck` for every combo, returns governing UC.

In [ ]:
def check_single_stud(cfs, section_path, combos, lengths, spec=SPEC):
    """
    Load a .cfss section and check it against all load combos.

    Parameters
    ----------
    cfs          : win32com CFS Calculation object
    section_path : str – full path to .cfss file
    combos       : list of dicts from build_load_combinations()
    lengths      : dict from unbraced_lengths()  {Lx, Ly, Lt, Lm}
    spec         : int – CFS spec enum (default AISI2024USASD = 42)

    Returns
    -------
    dict with:
      section, passed, governing_UC, governing_combo, governing_label,
      governing_forces, all_combos
    """
    cfs.LoadSection(section_path)

    # ── Build MemberParams (fixed, reused for all combos) ──
    mp = win32com.client.Dispatch(CFS_PROGID.replace('Calculation','MemberParams'))
    mp.Spec        = spec
    mp.Lx          = lengths['Lx']
    mp.Ly          = lengths['Ly']
    mp.Lt          = lengths['Lt']
    mp.Lm          = lengths['Lm']
    mp.Kx          = 1.0
    mp.Ky          = 1.0
    mp.Kt          = 1.0
    mp.Cbx         = 1.0
    mp.Cby         = 1.0
    mp.Cmx         = 1.0
    mp.Cmy         = 1.0
    mp.BracedFlange = 0    # Flange_None
    mp.R           = 0.0
    mp.Kf          = 0.0
    mp.Pdelta      = False

    forces = win32com.client.Dispatch(CFS_PROGID.replace('Calculation','SectionForces'))

    combo_results = []
    governing_UC  = 0.0
    governing_row = None

    for c in combos:
        forces.P  =  c['P_k']    # compression positive
        forces.Mx =  c['Mx_kin'] # compression on top positive
        forces.My =  0.0
        forces.Vx =  0.0
        forces.Vy =  c['Vy_k']
        forces.Dx =  0.0         # skip deflection check
        forces.Dy =  0.0

        chk = cfs.MemberCheck(mp, forces)

        # Governing unity check = max of combined checks
        UC = max(
            chk.PMxMy1 if chk.PMxMy1 is not None else 0,
            chk.PMxMy2 if chk.PMxMy2 is not None else 0,
            chk.MxVy   if chk.MxVy   is not None else 0,
        )

        row = {
            'combo':      c['combo'],
            'label':      c['label'],
            'P_k':        c['P_k'],
            'Mx_kin':     c['Mx_kin'],
            'Vy_k':       c['Vy_k'],
            'PMxMy1':     round(chk.PMxMy1, 4) if chk.PMxMy1 else 0,
            'PMxMy2':     round(chk.PMxMy2, 4) if chk.PMxMy2 else 0,
            'MxVy':       round(chk.MxVy,   4) if chk.MxVy   else 0,
            'UC':         round(UC, 4),
            'status':     'PASS' if UC <= 1.0 else 'FAIL',
        }
        combo_results.append(row)

        if UC > governing_UC:
            governing_UC  = UC
            governing_row = row

    return {
        'section':          section_path.split('\\')[-1].replace('.cfss',''),
        'passed':           governing_UC <= 1.0,
        'governing_UC':     round(governing_UC, 4),
        'governing_combo':  governing_row['combo'],
        'governing_label':  governing_row['label'],
        'governing_forces': {
            'P_k':    governing_row['P_k'],
            'Mx_kin': governing_row['Mx_kin'],
            'Vy_k':   governing_row['Vy_k'],
        },
        'all_combos': combo_results,
    }

## 6. Wall design: iterate sections, pick lightest that passes

In [ ]:
def design_wall_stud(
    # ── Geometry ──
    wall_height_ft  = 10.0,
    stud_spacing_in = 16.0,
    trib_A_ft       = 10.0,   # half-span of supported member, side A
    trib_B_ft       = 0.0,    # 0 for exterior walls
    brace_config    = 'third',# 'none','midpoint','third','48in','60in','72in','custom'
    custom_n        = None,   # used if brace_config='custom'
    wall_type       = 'exterior',

    # ── ASD Loads (PSF) ──
    D_wall   = 12.0,   # wall self-weight
    D_A      = 80.0,   # dead load side A
    L_A      = 80.0,   # live load side A
    D_B      = 0.0,    # dead load side B (interior only)
    L_B      = 0.0,    # live load side B (interior only)
    S        = 25.0,   # roof snow
    W_uplift = 20.0,   # roof wind uplift
    W_mwfw   = 15.0,   # MWFRS windward lateral
    W_mwfl   = 15.0,   # MWFRS leeward lateral
    W_cc_w   = 30.0,   # C&C windward lateral
    W_cc_l   = 30.0,   # C&C leeward lateral
    L_lat    = 0.0,    # interior lateral live
    Eh       = 0.3,    # seismic horizontal coefficient
    Ev       = 0.14,   # seismic vertical coefficient

    # ── Section list (.cfss paths, lightest → heaviest) ──
    sections = None,
    spec     = SPEC,
):
    """
    Design a wall stud: tries each section in order, returns first that passes.
    sections: list of full .cfss paths ordered lightest to heaviest.
    """
    if sections is None:
        raise ValueError('Provide a list of .cfss section paths.')

    geometry = {
        'height_ft':   wall_height_ft,
        'spacing_in':  stud_spacing_in,
        'trib_A_ft':   trib_A_ft,
        'trib_B_ft':   trib_B_ft,
        'wall_type':   wall_type,
    }

    loads = {
        'D_wall':   D_wall,
        'D_A':      D_A,    'L_A':    L_A,
        'D_B':      D_B,    'L_B':    L_B,
        'S':        S,      'W_uplift': W_uplift,
        'W_mwfw':   W_mwfw, 'W_mwfl':  W_mwfl,
        'W_cc_w':   W_cc_w, 'W_cc_l':  W_cc_l,
        'L_lat':    L_lat,
        'Eh':       Eh,     'Ev':      Ev,
    }

    combos  = build_load_combinations(loads, geometry)
    lengths = unbraced_lengths(wall_height_ft, brace_config, custom_n)
    cfs     = get_cfs()

    all_section_results = []

    for sec_path in sections:
        try:
            result = check_single_stud(cfs, sec_path, combos, lengths, spec)
            all_section_results.append(result)
            if result['passed']:
                result['all_sections_tried'] = all_section_results
                return result
        except Exception as e:
            print(f'  Error on {sec_path}: {e}')
            continue

    # Nothing passed
    return {
        'section':        None,
        'passed':         False,
        'governing_UC':   None,
        'message':        'No section in the list passes. Increase section size or reduce loads.',
        'all_sections_tried': all_section_results,
    }

## 7. Example wall — Simpson CFS comparison values

Exterior wall, 10 ft height, 16" spacing, third-point bridging.

In [ ]:
# ── Section library paths ── modify to match your system ──
SECTION_DIR = r'C:\path\to\MSBS_Sections\SINGLE'   # <-- UPDATE THIS

SECTIONS_EXTERIOR = [
    SECTION_DIR + r'\362SLS155-33_20_GA.cfss',
    SECTION_DIR + r'\362SLS155-43_18_GA.cfss',
    SECTION_DIR + r'\362SLS155-54_16_GA.cfss',
    SECTION_DIR + r'\600SLS243-33_20_GA.cfss',
    SECTION_DIR + r'\600SLS243-43_18_GA.cfss',
    SECTION_DIR + r'\600SLS243-54_16_GA.cfss',
    SECTION_DIR + r'\600SLS243-68_14_GA.cfss',
    SECTION_DIR + r'\984SLS295-33_20_GA.cfss',
    SECTION_DIR + r'\984SLS295-43_18_GA.cfss',
    SECTION_DIR + r'\984SLS295-54_16_GA.cfss',
    SECTION_DIR + r'\984SLS295-68_14_GA.cfss',
]

# ── Run the design ──
result = design_wall_stud(
    # Geometry
    wall_height_ft  = 10.0,
    stud_spacing_in = 16.0,
    trib_A_ft       = 10.0,
    trib_B_ft       = 0.0,
    brace_config    = 'third',
    wall_type       = 'exterior',

    # Loads (PSF) — from your spec
    D_wall   = 12.0,
    D_A      = 80.0,
    L_A      = 80.0,
    S        = 25.0,
    W_uplift = 20.0,
    W_mwfw   = 15.0,
    W_mwfl   = 15.0,
    W_cc_w   = 30.0,
    W_cc_l   = 30.0,
    Eh       = 0.3,
    Ev       = 0.14,

    sections = SECTIONS_EXTERIOR,
)

## 8. Results display

In [ ]:
def display_result(result):
    if not result['passed'] and result['section'] is None:
        print('FAIL:', result.get('message'))
        return

    status = 'PASS' if result['passed'] else 'FAIL'
    print(f"{'='*55}")
    print(f"  {status}  →  {result['section']}")
    print(f"  Governing UC : {result['governing_UC']} (combo {result['governing_combo']})")
    print(f"  Combo label  : {result['governing_label']}")
    f = result['governing_forces']
    print(f"  Forces       : P={f['P_k']} k   Mx={f['Mx_kin']} k·in   Vy={f['Vy_k']} k")
    print(f"{'='*55}")

    # ── Combo detail table ──
    df = pd.DataFrame(result['all_combos'])[
        ['combo','label','P_k','Mx_kin','Vy_k','PMxMy1','PMxMy2','MxVy','UC','status']
    ]
    display(df.style
              .applymap(lambda v: 'background-color:#ffd0d0' if v == 'FAIL' else
                                  'background-color:#d0ffd0' if v == 'PASS' else '',
                        subset=['status'])
              .format({'P_k':'{:.3f}','Mx_kin':'{:.2f}','Vy_k':'{:.4f}',
                       'PMxMy1':'{:.3f}','PMxMy2':'{:.3f}','MxVy':'{:.3f}','UC':'{:.3f}'}))

    # ── Sections tried table ──
    tried = result.get('all_sections_tried', [])
    if tried:
        rows = [{'section': r['section'],
                 'governing_UC': r['governing_UC'],
                 'combo': r['governing_combo'],
                 'status': 'PASS' if r['passed'] else 'FAIL'}
                for r in tried]
        print('\nSections tried:')
        display(pd.DataFrame(rows).style
                  .applymap(lambda v: 'background-color:#ffd0d0' if v == 'FAIL' else
                                      'background-color:#d0ffd0' if v == 'PASS' else '',
                             subset=['status']))


display_result(result)

## 9. Load combination preview (forces only, no CFS needed)

In [ ]:
# Check the forces before running CFS — useful for debugging
geometry_preview = {
    'height_ft':  10.0,
    'spacing_in': 16.0,
    'trib_A_ft':  10.0,
    'trib_B_ft':  0.0,
    'wall_type':  'exterior',
}
loads_preview = {
    'D_wall': 12.0, 'D_A': 80.0, 'L_A': 80.0,
    'D_B': 0.0, 'L_B': 0.0,
    'S': 25.0, 'W_uplift': 20.0,
    'W_mwfw': 15.0, 'W_mwfl': 15.0,
    'W_cc_w': 30.0, 'W_cc_l': 30.0,
    'L_lat': 0.0, 'Eh': 0.3, 'Ev': 0.14,
}

combos_preview = build_load_combinations(loads_preview, geometry_preview)
df_combos = pd.DataFrame(combos_preview)
print('Load combinations (forces per stud):')
display(df_combos[['combo','label','P_k','Mx_kin','Vy_k']]
        .style.format({'P_k':'{:.3f}','Mx_kin':'{:.2f}','Vy_k':'{:.4f}'}))

## 10. Cleanup

In [ ]:
# Release network license when done
_cfs.ReleaseLicense()
_cfs = None
print('License released.')